In [5]:
%%time

!python ../scripts/analyze_tuned_results.py --results-dir ../results/ours --output-dir ../results/ours --update-tex --tex-file ../docs/main.tex

HYPERPARAMETER TUNED ML MODEL RESULTS ANALYSIS
Results directory: ../results/ours
Output directory: ../results/ours
Paper directory: ../results

Loaded 295 result rows from new tuning scripts
Stocks: ['AAPL', 'META', 'NVDA', 'SPY', 'TSLA']
Models: ['LSTM', 'XGBoost', 'LightGBM', 'RandomForest', 'GradientBoosting', 'LogisticRegression', 'SVM']
Strategies: ['long_short']
Horizons: [2, 3, 4, 5, 6, 7, 8, 9, 10]

ANALYSIS BY MODEL

--- LSTM ---
  Samples: 45
  Avg Accuracy:    0.5024 (+/- 0.0451)
  Avg ROC-AUC:     0.5102 (+/- 0.0796)
  Avg Trades:      53.2
  Avg Win Rate:    0.4285 (42.8%)
  Avg Sharpe:      -0.8435
  Avg Total Return: -0.1156 (-11.56%)

--- XGBoost ---
  Samples: 45
  Avg Accuracy:    0.4878 (+/- 0.0386)
  Avg ROC-AUC:     0.5216 (+/- 0.0411)
  Avg Trades:      53.2
  Avg Win Rate:    0.4210 (42.1%)
  Avg Sharpe:      -0.9195
  Avg Total Return: 0.0740 (7.40%)

--- LightGBM ---
  Samples: 45
  Avg Accuracy:    0.4892 (+/- 0.0407)
  Avg ROC-AUC:     0.5249 (+/- 0.0462)
  

In [6]:
import pandas as pd

df = pd.read_csv("../results/ours/tuned_summary_by_model_strategy.csv")
df

,Model,Strategy,Samples,Avg_Accuracy,Avg_ROC_AUC,Avg_Trades,Avg_WinRate,Avg_Sharpe,Avg_TotalReturn
0,LSTM,long_short,45,0.502363,0.510234,53.222222,0.428466,-0.843451,-0.115630
1,XGBoost,long_short,45,0.487826,0.521611,53.222222,0.420971,-0.919466,0.074027
2,LightGBM,long_short,45,0.489243,0.524864,53.222222,0.430207,-0.760367,0.058579
3,RandomForest,long_short,45,0.487649,0.546532,53.222222,0.416464,-0.934778,0.053820
4,GradientBoosting,long_short,45,0.483931,0.533538,53.222222,0.436846,-0.733870,0.114217
5,LogisticRegression,long_short,25,0.493705,0.649939,41.800000,0.397290,-1.157677,-0.038610
6,SVM,long_short,45,0.497742,0.617353,53.222222,0.408075,-1.039945,0.051723


In [37]:
df[df['Avg_TotalReturn'] > 0].sort_values(by='Avg_TotalReturn', ascending=False)

,Model,Strategy,Samples,Avg_Accuracy,Avg_ROC_AUC,Avg_Trades,Avg_WinRate,Avg_Sharpe,Avg_TotalReturn
25,SVM,long_only,21,0.483779,0.578293,6.619048,0.377414,0.633334,0.162829
19,GradientBoosting,long_only_confidence,21,0.480744,0.500254,8.047619,0.290504,0.402751,0.154456
5,XGBoost,long_only,21,0.479416,0.489435,5.380952,0.164164,0.285454,0.152477
13,RandomForest,long_only,21,0.476760,0.509287,8.047619,0.356398,0.583058,0.145349
9,LightGBM,long_only,21,0.477329,0.495740,8.047619,0.220657,0.354642,0.121806
17,GradientBoosting,long_only,21,0.480744,0.500254,11.095238,0.392478,0.574670,0.117833
16,GradientBoosting,long_short,45,0.483931,0.533538,53.222222,0.436846,-0.733870,0.114217
1,LSTM,long_only,21,0.498815,0.546656,8.952381,0.309152,0.525888,0.105088
21,LogisticRegression,long_only,21,0.480554,0.617767,1.476190,0.204649,0.337767,0.097442
4,XGBoost,long_short,45,0.487826,0.521611,53.222222,0.420971,-0.919466,0.074027


In [7]:
# Per-stock Best Model Selection (for main.tex Tables 1 & 3)
# This is the unified methodology used throughout the paper

import pandas as pd

df_all = pd.read_csv('../results/ours/tuned_all_results_combined.csv')
df_ls = df_all[df_all['Strategy'] == 'long_short']

# ============================================================
# Best by AUC: For each stock, select best model by AUC
# ============================================================
print("=" * 90)
print("BEST BY AUC (Direct Per-Stock Selection)")
print("=" * 90)
print(f"{'Stock':<6} {'Model':<20} {'h':<3} {'Acc':<6} {'AUC':<6} {'N':<4} {'Win%':<6} {'Sharpe':<8} {'Ret%':<8}")
print("-" * 90)

best_auc_rows = []
for stock in ['AAPL', 'META', 'NVDA', 'SPY', 'TSLA']:
    stock_data = df_ls[df_ls['Stock'] == stock]
    if not stock_data.empty:
        best_idx = stock_data['Test_ROC_AUC'].idxmax()
        row = stock_data.loc[best_idx]
        best_auc_rows.append(row)
        model_name = 'LR' if row['Model'] == 'LogisticRegression' else row['Model']
        print(f"{stock:<6} {model_name:<20} {int(row['Horizon']):<3} {row['Test_Accuracy']:.3f} {row['Test_ROC_AUC']:.3f} {int(row['Trades']):<4} {row['WinRate']*100:.1f}  {row['Sharpe']:.2f}    {row['TotalReturn']*100:.1f}")

print("-" * 90)
best_auc_df = pd.DataFrame(best_auc_rows)
print(f"{'Avg':<6} {'':<20} {best_auc_df['Horizon'].mean():.1f} {best_auc_df['Test_Accuracy'].mean():.3f} {best_auc_df['Test_ROC_AUC'].mean():.3f} {best_auc_df['Trades'].mean():.0f}   {best_auc_df['WinRate'].mean()*100:.1f}  {best_auc_df['Sharpe'].mean():.2f}    {best_auc_df['TotalReturn'].mean()*100:.1f}")

# ============================================================
# Best by Sharpe: For each stock, select best model by Sharpe
# ============================================================
print("\n" + "=" * 90)
print("BEST BY SHARPE (Direct Per-Stock Selection)")
print("=" * 90)
print(f"{'Stock':<6} {'Model':<20} {'h':<3} {'Acc':<6} {'AUC':<6} {'N':<4} {'Win%':<6} {'Sharpe':<8} {'Ret%':<8}")
print("-" * 90)

best_sharpe_rows = []
for stock in ['AAPL', 'META', 'NVDA', 'SPY', 'TSLA']:
    stock_data = df_ls[df_ls['Stock'] == stock]
    if not stock_data.empty:
        best_idx = stock_data['Sharpe'].idxmax()
        row = stock_data.loc[best_idx]
        best_sharpe_rows.append(row)
        model_name = 'LR' if row['Model'] == 'LogisticRegression' else row['Model']
        model_name = 'GB' if row['Model'] == 'GradientBoosting' else model_name
        print(f"{stock:<6} {model_name:<20} {int(row['Horizon']):<3} {row['Test_Accuracy']:.3f} {row['Test_ROC_AUC']:.3f} {int(row['Trades']):<4} {row['WinRate']*100:.1f}  {row['Sharpe']:.2f}    {row['TotalReturn']*100:.1f}")

print("-" * 90)
best_sharpe_df = pd.DataFrame(best_sharpe_rows)
print(f"{'Avg':<6} {'':<20} {best_sharpe_df['Horizon'].mean():.1f} {best_sharpe_df['Test_Accuracy'].mean():.3f} {best_sharpe_df['Test_ROC_AUC'].mean():.3f} {best_sharpe_df['Trades'].mean():.0f}   {best_sharpe_df['WinRate'].mean()*100:.1f}  {best_sharpe_df['Sharpe'].mean():.2f}    {best_sharpe_df['TotalReturn'].mean()*100:.1f}")

print("\n" + "=" * 90)
print("SUMMARY FOR MAIN.TEX TABLE 3")
print("=" * 90)
print(f"Best by AUC:    Acc={best_auc_df['Test_Accuracy'].mean():.3f}, AUC={best_auc_df['Test_ROC_AUC'].mean():.3f}, N={best_auc_df['Trades'].mean():.0f}, Win%={best_auc_df['WinRate'].mean()*100:.1f}, Sharpe={best_auc_df['Sharpe'].mean():.2f}, Ret%={best_auc_df['TotalReturn'].mean()*100:.1f}")
print(f"Best by Sharpe: Acc={best_sharpe_df['Test_Accuracy'].mean():.3f}, AUC={best_sharpe_df['Test_ROC_AUC'].mean():.3f}, N={best_sharpe_df['Trades'].mean():.0f}, Win%={best_sharpe_df['WinRate'].mean()*100:.1f}, Sharpe={best_sharpe_df['Sharpe'].mean():.2f}, Ret%={best_sharpe_df['TotalReturn'].mean()*100:.1f}")

BEST BY AUC (Direct Per-Stock Selection)
Stock  Model                h   Acc    AUC    N    Win%   Sharpe   Ret%    
------------------------------------------------------------------------------------------
AAPL   LR                   8   0.566 0.557 31   48.4  -1.67    -30.0
META   LR                   10  0.490 0.774 25   32.0  -1.44    -43.1
NVDA   LR                   8   0.418 0.793 31   22.6  -2.12    -66.6
SPY    LR                   8   0.442 0.754 31   32.3  -3.06    -24.8
TSLA   LR                   5   0.566 0.606 50   56.0  1.85    157.5
------------------------------------------------------------------------------------------
Avg                         7.8 0.496 0.697 34   38.2  -1.29    -1.4

BEST BY SHARPE (Direct Per-Stock Selection)
Stock  Model                h   Acc    AUC    N    Win%   Sharpe   Ret%    
------------------------------------------------------------------------------------------
AAPL   LightGBM             9   0.534 0.500 27   74.1  1.40    29.0
MET